# Caught & Shared — MetaPath-based Approach
**Preserving Bipartite Structure with Explicit Mentor Influence**

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch_geometric.data import HeteroData
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

np.random.seed(42)
torch.manual_seed(42)

#### 1. Synthetic Data

In [ ]:
num_users = 25
num_items = 80

# User-item interactions
interactions = []
for u in range(num_users):
    num_inter = np.random.randint(4, 12)
    items = np.random.choice(num_items, num_inter, replace=False)
    for i in items:
        interactions.append((u, i))

inter_df = pd.DataFrame(interactions, columns=['user_id', 'item_id'])

# Mentor assignments (user can have multiple mentors)
multi_mentor_edges = [
    (0, 5, 0.70),
    (0, 7, 0.30),
    (1, 12, 0.85),
    (2, 5, 0.65),
    (3, 8, 0.90),
    (4, 15, 0.55),
    (4, 9, 0.45),
]

print(f"{len(inter_df)} user-item interactions")
print(f"{len(multi_mentor_edges)} mentor assignments")

#### 2. MetaPath-based Aggregation Functions

In [ ]:
def get_mentor_items(mentor_id, inter_df):
    """Items interacted by mentor"""
    return inter_df[inter_df['user_id'] == mentor_id]['item_id'].tolist()


def get_user_items(user_id, inter_df):
    """Items interacted by user"""
    return inter_df[inter_df['user_id'] == user_id]['item_id'].tolist()


def metapath_aggregation(user_emb, 
                        mentor_embs, 
                        alphas, 
                        item_embs, 
                        user_items, 
                        mentor_items_lists,
                        weights=None):
    
    if weights is None:
        weights = {'U_M_I': 0.5, 'U_I_M_I': 0.5}
    
    final_emb = user_emb.clone()
    
    for mentor_emb, alpha, mentor_items in zip(mentor_embs, alphas, mentor_items_lists):
        if len(mentor_items) == 0:
            continue
            
        # MetaPath 1: U → M → I
        # Direct preference of the mentor
        if len(mentor_items) > 0:
            mentor_pref = item_embs[mentor_items].mean(dim=0, keepdim=True)
            final_emb += alpha * weights['U_M_I'] * (mentor_pref - user_emb)
        
        # MetaPath 2: U → I ← M → I
        # Common interest + mentor's unique taste
        user_item_set = set(user_items)
        mentor_item_set = set(mentor_items)
        
        common_items = list(user_item_set & mentor_item_set)
        mentor_unique = list(mentor_item_set - user_item_set)
        
        if common_items:
            common_emb = item_embs[common_items].mean(dim=0, keepdim=True)
            final_emb += alpha * weights['U_I_M_I'] * 0.6 * (common_emb - user_emb)
        
        if mentor_unique:
            unique_pref = item_embs[mentor_unique].mean(dim=0, keepdim=True)
            final_emb += alpha * weights['U_I_M_I'] * 0.4 * (unique_pref - user_emb)
    
    return final_emb

In [ ]:
@torch.no_grad()
def get_recommendations_metapath(user_emb, 
                                 item_embs, 
                                 user_id, 
                                 inter_df, 
                                 mentor_config=None, 
                                 k=8):
    """    
    Parameters:
        user_emb: Tensor - embedding of the target user
        item_embs: Tensor - embeddings of all items [num_items, emb_dim]
        user_id: int - target user
        inter_df: DataFrame - user-item interactions
        mentor_config: list of tuples [(mentor_id, alpha), ...] or None
        k: number of recommendations to return
    """
    if mentor_config is None or len(mentor_config) == 0:
        final_emb = user_emb
    else:
        mentor_ids = [m for m, a in mentor_config]
        alphas = [a for m, a in mentor_config]
        
        mentor_embs = torch.stack([user_embs[m] for m in mentor_ids])  # assuming user_embs is defined globally or passed
        
        # Get items for user and each mentor
        user_items = get_user_items(user_id, inter_df)
        mentor_items_lists = [get_mentor_items(m, inter_df) for m in mentor_ids]
        
        final_emb = metapath_aggregation(
            user_emb=user_emb,
            mentor_embs=mentor_embs,
            alphas=alphas,
            item_embs=item_embs,
            user_items=user_items,
            mentor_items_lists=mentor_items_lists
        )
    
    scores = torch.matmul(final_emb, item_embs.T).squeeze(0)
    
    topk_scores, topk_indices = torch.topk(scores, k=k)
    
    return {
        'user_id': user_id,
        'mentors': mentor_config,
        'recommended_items': topk_indices.tolist(),
        'scores': topk_scores.tolist()
    }

#### MetaPath Strategy in Caught & Shared

In the MetaPath-based approach we do not add direct `user → mentor` edges, preserving the original bipartite structure. Instead, we define meaningful **metapaths** that allow the model to leverage mentor information:

##### Used MetaPaths:

1. **U → M → I** (User → Mentor → Item)  
   The most direct path. It captures the mentor’s general taste and preferences.

2. **U → I ← M → I** (Common + Unique Interests)  
   This path is particularly powerful — it finds:
   - Items liked by both the user and the mentor (common ground)
   - Unique items that the mentor likes but the user hasn't seen yet (serendipity)

3. **U → M → U → I**
   Can be used, not considered here.

##### Why this is powerful for Caught & Shared:

- The system respects the user’s own history while intentionally incorporating the mentor’s taste.
- It creates a natural balance between **familiarity** and **discovery**.
- Multiple mentors can be combined with different influence weights (`alpha`).
- The approach remains fully compatible with any bipartite GNN or Two-Tower model.

> *MetaPath aggregation allows us to inject human taste into recommendations without breaking the fundamental graph structure.*